In [ ]:
import os
import shutil
import numpy as np
from config import Config
from env import MySumoEnv

_NEEDED_WORKER_FILES = ("detectors.add.xml", "network.net.xml", "run.sumocfg")

def prepare_worker_dir(worker_idx=0):
    if hasattr(Config, "ensure_dirs"):
        Config.ensure_dirs()
    else:
        os.makedirs(Config.WORKERS_ROOT, exist_ok=True)
    src = Config.BASE_WORK_DIR
    dst = Config.worker_dir(worker_idx)
    os.makedirs(dst, exist_ok=True)
    for name in _NEEDED_WORKER_FILES:
        src_path = os.path.join(src, name)
        dst_path = os.path.join(dst, name)
        if not os.path.exists(src_path):
            raise FileNotFoundError(f"No utils file: {src_path}")
        shutil.copy2(src_path, dst_path)
    os.makedirs(os.path.join(dst, "dump"), exist_ok=True)
    return dst

def build_env(worker_idx=0, total_step=None, seed=42):
    rl_dir = prepare_worker_dir(worker_idx)
    answer_path = Config.answer_path_for(worker_idx)
    sumocfg_path = os.path.join(rl_dir, "run.sumocfg")

    if not os.path.exists(rl_dir):
        raise FileNotFoundError(f"No directory: {rl_dir}")
    if not os.path.exists(sumocfg_path):
        raise FileNotFoundError(f"No run.sumocfg: {sumocfg_path}")
    if not os.path.exists(answer_path):
        raise FileNotFoundError(f"No answer.csv: {answer_path}")

    if total_step is None:
        total_step = Config.TOTAL_STEP

    env = MySumoEnv(
        rl_dir=rl_dir,
        sumo_binary=Config.SUMO_BINARY,
        origin_list=Config.ORIGIN_LIST,
        destination_list=Config.DESTINATION_LIST,
        input_interval=Config.INPUT_INTERVAL,
        detector_interval=Config.DETECTOR_INTERVAL,
        num_OD=Config.NUM_OD,
        state_dim=1,
        answer_dir=answer_path,
        total_step=int(total_step),
    )

    obs, info = env.reset(seed=seed)
    return env, obs, info

In [ ]:

import os
import json
import numpy as np
from bayes_opt import BayesianOptimization
from config import Config

def run_episode_with_plan(build_env_fn, worker_idx, action_plan, seed):
    env, obs, info = build_env_fn(worker_idx=worker_idx, total_step=action_plan.shape[0], seed=seed)
    total_reward = 0.0
    last_info = {}

    try:
        for t in range(action_plan.shape[0]):
            obs, reward, terminated, truncated, info = env.step(action_plan[t])
            total_reward += float(reward)
            last_info = info
            if terminated or truncated:
                break
    finally:
        env.close()

    trajectory = last_info.get("trajectory", [])
    return float(total_reward), trajectory

def _decode_block_binary(params: dict, n_local: int, num_od: int) -> np.ndarray:
\
\
\
       
    n_var = n_local * num_od
    x = np.array([params[f"x{i}"] for i in range(n_var)], dtype=np.float32)
    block_action = (x >= 0.5).astype(np.int32).reshape(n_local, num_od)
    return block_action

def sequential_bayes_optimize_binary_300s(
    build_env_fn,
    worker_idx,
    num_od,
    total_step,
    input_interval,
    detector_interval,
    init_points=2,
    n_iter=8,
    seed=42,
    eval_seed=None,
    verbose=2,
):
\
\
\
\
\
       
    block_steps = detector_interval // input_interval
    n_blocks = int(np.ceil(total_step / block_steps))

    fixed_actions = np.zeros((0, num_od), dtype=np.int32)
    block_logs = []
    eval_count = {"n": 0}

    for b in range(n_blocks):
        s = b * block_steps
        e = min((b + 1) * block_steps, total_step)
        n_local = e - s
        n_var = n_local * num_od

        if b == 0:
            prefix_reward = 0.0
        else:
            prefix_reward, _ = run_episode_with_plan(
                build_env_fn=build_env_fn,
                worker_idx=worker_idx,
                action_plan=fixed_actions,
                seed=seed + eval_count["n"],
            )
            eval_count["n"] += 1

        pbounds = {f"x{i}": (0.0, 1.0) for i in range(n_var)}

        def objective(**kwargs):
            cur_block_action = _decode_block_binary(kwargs, n_local=n_local, num_od=num_od)

            if fixed_actions.shape[0] == 0:
                candidate_plan = cur_block_action
            else:
                candidate_plan = np.vstack([fixed_actions, cur_block_action])

            total_reward_to_e, _ = run_episode_with_plan(
                build_env_fn=build_env_fn,
                worker_idx=worker_idx,
                action_plan=candidate_plan,
                seed=seed + eval_count["n"],
            )
            eval_count["n"] += 1

            return float(total_reward_to_e - prefix_reward)

        bo = BayesianOptimization(
            f=objective,
            pbounds=pbounds,
            random_state=seed + b,
            verbose=verbose,
            allow_duplicate_points=True,
        )
        bo.maximize(init_points=init_points, n_iter=n_iter)

        best_block_action = _decode_block_binary(bo.max["params"], n_local=n_local, num_od=num_od)

        if fixed_actions.shape[0] == 0:
            fixed_actions = best_block_action
        else:
            fixed_actions = np.vstack([fixed_actions, best_block_action])

        committed_reward_to_e, _ = run_episode_with_plan(
            build_env_fn=build_env_fn,
            worker_idx=worker_idx,
            action_plan=fixed_actions,
            seed=seed + eval_count["n"],
        )
        eval_count["n"] += 1

        block_logs.append({
            "block": int(b),
            "step_start": int(s),
            "step_end": int(e),
            "n_local": int(n_local),
            "n_var": int(n_var),
            "best_block_objective": float(bo.max["target"]),
            "committed_reward_to_block_end": float(committed_reward_to_e),
            "best_block_action_sum_per_od": best_block_action.sum(axis=0).astype(int).tolist(),
        })

        print(
            f"[block {b}] steps {s}~{e} | n_var={n_var} | "
            f"action_sum_per_od={best_block_action.sum(axis=0).tolist()} | "
            f"block_obj={bo.max['target']:.6f} | committed_R={committed_reward_to_e:.6f}"
        )

    best_actions = fixed_actions[:total_step].astype(np.int32)

    replay_seed = seed if eval_seed is None else eval_seed
    final_reward, final_traj = run_episode_with_plan(
        build_env_fn=build_env_fn,
        worker_idx=worker_idx,
        action_plan=best_actions,
        seed=replay_seed,
    )

    return best_actions, final_reward, final_traj, block_logs

In [ ]:
import time
from trial_timing import reset_trial_times, append_trial_time
os.makedirs(Config.RESULT_DIR, exist_ok=True)

_trial_result_dir = RESULT_DIR if "RESULT_DIR" in globals() else Config.RESULT_DIR
reset_trial_times(_trial_result_dir)

for trial in range(5):
    _trial_t0 = time.perf_counter()
    trial_idx = trial
    seed_opt = int(100 + (trial_idx + 1))
                                                                          
    seed_eval = trial

    best_actions, final_reward, trajectory, block_logs = sequential_bayes_optimize_binary_300s(
        build_env_fn=build_env,
        worker_idx=0,
        num_od=Config.NUM_OD,
        total_step=Config.TOTAL_STEP,
        input_interval=Config.INPUT_INTERVAL,
        detector_interval=Config.DETECTOR_INTERVAL,
        init_points=0,
        n_iter=300,
        seed=seed_opt,
        eval_seed=seed_eval,
        verbose=2,
    )

    print(f"[trial {trial}] final_reward = {final_reward}")
    print(f"[trial {trial}] best_actions.shape = {best_actions.shape}")

    with open(os.path.join(Config.RESULT_DIR, f"trajectory_{trial}.json"), "w", encoding="utf-8") as f:
        json.dump(
            trajectory, f, ensure_ascii=False, indent=2,
            default=lambda o: o.tolist() if isinstance(o, np.ndarray) else o
        )

    with open(os.path.join(Config.RESULT_DIR, f"block_logs_{trial}.json"), "w", encoding="utf-8") as f:
        json.dump(block_logs, f, ensure_ascii=False, indent=2)
    append_trial_time(_trial_result_dir, trial, time.perf_counter() - _trial_t0)


In [ ]:
import os
import shutil
from config import Config


def remove_workers_root():
                                                            
    root = os.path.abspath(Config.WORKERS_ROOT)
    project_root = os.path.abspath(Config.RL_ROOT)
    if os.path.basename(root) != "workers":
        raise RuntimeError(f"Refusing to delete non-workers path: {root}")
    if os.path.commonpath([root, project_root]) != project_root:
        raise RuntimeError(f"Refusing to delete path outside experiment root: {root}")
    if os.path.isdir(root):
        shutil.rmtree(root)


remove_workers_root()
